# Set 4 오답노트

## 검토한 파일

- `set_01_06_answer/04_question.ipynb`
- `set_01_06_answer/04_answer.ipynb`
- `set_01_06_answer/set_04_answer.ipynb`
- `set_01_06_answer/set_04_cbi.ipynb`
- `00_trying/01/04_question.ipynb`
- `00_trying/02/04_question.ipynb`

## 최종 답

- Q01: 직업 번호 **4**
- Q02: 결혼 여부별 평균 차이 **0.13**
- Q03: Silhouette score **0.18**

현재 `00_trying/02/04_question.ipynb`의 수정된 풀이는 Q1 `4`, Q2 `0.13`, Q3 `0.18`로 모두 맞다. Q3의 이전 코드에는 `총 구매금액`이 빠지고 고객 식별자 `user`가 군집변수에 들어가 `0.17`이 나오는 오류가 있었으나, 현재는 `purchase=('purchase', 'sum')`으로 집계하고 `user`를 인덱스로 유지하여 수정되었다. 아래 내용은 이전 오류의 원인과 현재 수정 코드가 맞는 이유를 함께 정리한다.


## Q01 — 매출액이 가장 큰 상품과 최다 구매 직업

문제에는 서로 다른 두 집계 기준이 등장한다.

1. 가장 큰 상품을 찾을 때는 상품별 **매출액 합계**를 사용한다.
2. 그 상품을 가장 많이 구매한 직업을 찾을 때는 직업별 **구매 기록 개수**를 사용한다.

```python
sales_by_product = df.groupby('prod')['purchase'].sum()
best_product = sales_by_product.idxmax()

best_job = (
    df.loc[df['prod'] == best_product, 'job']
    .value_counts()
    .idxmax()
)
display(best_job)  # 4
```

`sales_by_product.max()`는 최대 매출액 값 `27,995,166`을 반환하고, `sales_by_product.idxmax()`는 그 값의 인덱스인 상품 코드 `P00025442`를 반환한다. 문제는 상품 코드가 필요하므로 `idxmax()`를 사용한다.

두 번째 단계에서 `purchase.sum()`을 다시 사용하면 직업별 구매금액을 비교하게 된다. 문제는 구매 **개수**를 기준으로 하라고 했으므로 행의 빈도를 세는 `value_counts()`가 맞다.


## Q02 — 고객별 카테고리 개수와 결혼 여부별 평균

### 이전 Q2에서 Q1의 DataFrame을 참조했던 실수

이전 풀이에는 다음 코드가 있었다.

```python
df_q2_1 = df_q1[cols_q2].copy()
```

Q2에서는 `df_q2`를 사용해야 한다. 당시 `df_q1`과 `df_q2`가 모두 원본의 복사본이어서 결과 `0.13`은 같았지만, Q1에서 행이나 값을 변경하면 Q2까지 영향을 받을 수 있었다. 현재 풀이에서는 다음과 같이 이미 수정되었다.

```python
df_q2_1 = df_q2[cols_q2].copy()
```

### 카테고리 조합 만들기

결측치는 카테고리 열에만 0으로 채우고, `1.0`이 아닌 `1` 형태의 문자열을 만들기 위해 정수로 바꾼 다음 문자열로 변환한다.

```python
cat_cols = ['prod_cat1', 'prod_cat2', 'prod_cat3']
df_q2 = df.loc[df['age_group'] == '26-35'].copy()
df_q2[cat_cols] = df_q2[cat_cols].fillna(0).astype(int).astype(str)
df_q2['prod_cat'] = (
    df_q2['prod_cat1'] + '-'
    + df_q2['prod_cat2'] + '-'
    + df_q2['prod_cat3']
)
```

숫자 상태에서 `+`를 사용하면 카테고리 번호가 더해진다. 문자열로 변환한 뒤 이어 붙여야 `1-2-0`과 같은 하나의 조합이 된다.

### 고객 단위에서 센 뒤 결혼 여부 단위로 평균 내기

```python
user_counts = (
    df_q2.groupby(['user', 'marital'])['prod_cat']
    .nunique()
    .reset_index(name='category_count')
)

marital_mean = user_counts.groupby('marital')['category_count'].mean()
answer_q2 = round(abs(marital_mean.loc[0] - marital_mean.loc[1]), 2)
display(answer_q2)  # 0.13
```

원본 데이터에서 한 행은 구매 거래 한 건이다. 결혼 여부별로 원본 행을 바로 집계하면 거래가 많은 고객이 더 큰 가중치를 갖는다. 문제는 **각 고객의 카테고리 개수**를 먼저 구한 다음 결혼 여부별 고객 평균을 구하라고 했으므로 집계가 두 단계다.

### `reset_index(name='category_count')`에서 열이 생기는 원리

`groupby(['user', 'marital'])['prod_cat'].nunique()`의 결과는 다음 구조의 Series다.

- 인덱스 이름: `user`, `marital`
- Series 값: 고객별 고유 카테고리 개수
- Series의 기존 이름: `prod_cat`

`reset_index(name='category_count')`를 호출하면 인덱스였던 `user`, `marital`이 일반 열이 되고, Series 값은 사용자가 지정한 `category_count` 열이 된다. `prod_cat` 열이 저절로 생긴 것이 아니라 `name=`으로 집계값 열의 이름을 정한 것이다.

고객마다 `marital`이 하나로 고정되어 있으므로 `groupby(['user', 'marital'])`가 가능하다. 이를 확인하려면 다음처럼 고객별 고유값 개수의 최댓값이 1인지 검사할 수 있다.

```python
df.groupby('user')['marital'].nunique().max()
# 1
```


## Q03 핵심 — `groupby().agg()` 상세 설명

### 1. 왜 고객 단위 집계가 필요한가?

원본 `sales_pos.csv`는 **550,068개의 거래 행**으로 구성된다. 문제는 **5,891명의 고객**을 군집화하라고 했으므로 한 고객의 여러 거래를 고객 한 행으로 줄여야 한다.

```text
거래 단위 데이터: 550,068행
        ↓ groupby('user').agg(...)
고객 단위 데이터:   5,891행
```

`groupby('user')`는 같은 고객 번호를 가진 거래들을 하나의 그룹으로 묶고, `agg()`는 각 그룹을 어떤 대표값 하나로 줄일지 열마다 지정한다.

### 2. Named Aggregation 문법

```python
결과열이름=('원본열이름', '집계함수')
```

실제 코드:

```python
df_user = df.groupby('user').agg(
    gender=('gender', 'first'),
    age_group=('age_group', 'first'),
    job=('job', 'first'),
    city=('city', 'first'),
    marital=('marital', 'first'),
    prod_count=('prod', 'nunique'),
    total_purchase=('purchase', 'sum')
)
```

각 줄은 다음 의미다.

| 결과 열 | 원본 열 | 집계 함수 | 의미 |
|---|---|---|---|
| `gender` | `gender` | `first` | 고객의 성별 |
| `age_group` | `age_group` | `first` | 고객의 연령대 |
| `job` | `job` | `first` | 고객의 직업 |
| `city` | `city` | `first` | 고객의 도시 |
| `marital` | `marital` | `first` | 고객의 결혼 여부 |
| `prod_count` | `prod` | `nunique` | 고객이 구매한 서로 다른 상품 수 |
| `total_purchase` | `purchase` | `sum` | 고객의 모든 구매금액 합계 |

`first`, `nunique`, `sum`은 같은 고객 그룹에 서로 다른 의미를 적용한다.

### 3. `first`의 의미와 사용 조건

`first`는 각 그룹에서 해당 열의 첫 번째 결측치가 아닌 값을 가져온다. 성별·나이·직업·도시·결혼 여부가 고객마다 모든 거래에서 동일하다는 전제에서 고객의 대표값으로 사용할 수 있다.

```python
fixed_cols = ['gender', 'age_group', 'job', 'city', 'marital']
df.groupby('user')[fixed_cols].nunique().max()
```

각 결과가 모두 1이라면 고객별로 값이 고정되어 있다는 뜻이다. 값이 여러 개라면 단순히 `first`를 사용하기 전에 최신값, 최빈값 등 어떤 대표값이 필요한지 문제 의미를 판단해야 한다.

`first`는 정확히 첫 번째 행을 무조건 가져오는 `nth(0)`과 조금 다르다. 첫 행이 결측치라면 `first`는 다음의 결측치가 아닌 값을 선택한다.

### 4. `purchase=('purchase', 'first')`와 `sum`의 차이

`purchase`는 고객의 고정 속성이 아니라 거래마다 달라지는 구매금액이다. 따라서 어떤 집계함수를 쓰느냐에 따라 의미가 완전히 달라진다.

```python
purchase_first=('purchase', 'first')
```

이 코드는 고객의 **첫 번째 구매 거래 금액 한 건만** 가져온다. 총 구매금액이 아니다.

```python
total_purchase=('purchase', 'sum')
```

이 코드는 고객의 **모든 거래 금액을 합산**한다. 문제에서 요구하는 총 구매금액이다.

실제 데이터 비교:

| user | 첫 거래 `purchase` | `purchase` 합계 | 고유 상품 수 |
|---:|---:|---:|---:|
| 1 | 8,370 | 334,093 | 35 |
| 2 | 7,969 | 810,472 | 77 |

고객 1에게 `first`를 사용하면 총 구매금액이 `8,370`으로 표현되지만 실제 총 구매금액은 `334,093`이다. 고객 2도 `first`는 `7,969`, `sum`은 `810,472`다. `first`를 사용하면 고객의 소비 규모를 심하게 왜곡하여 군집 결과가 달라진다.

집계함수 선택 기준:

- 고객에게 고정된 속성 → `first`
- 모든 거래의 총액 → `sum`
- 서로 다른 상품 종류 수 → `nunique`
- 거래 횟수 → `size` 또는 `count`
- 거래당 평균 구매액 → `mean`

### 5. 집계 함수명 공백 오류

다음 코드는 오류가 발생한다.

```python
gender=('gender', ' first')
```

오류:

```text
AttributeError: 'SeriesGroupBy' object has no attribute ' first'
```

문자열 `' first'` 앞에 공백이 있어 pandas가 정확히 그 이름의 메서드를 찾기 때문이다. 집계 함수 문자열에는 불필요한 공백을 넣지 않는다.

```python
'first'    # 올바름
' first'   # 잘못됨
'first '   # 잘못됨
```

### 6. `user`를 인덱스로 둘지 열로 둘지

기본 `groupby('user').agg(...)` 결과에서 `user`는 인덱스다. 이 상태로 모델 입력을 만들면 고객 식별자가 독립변수에 들어가지 않는다.

```python
df_user = df.groupby('user').agg(...)
X = df_user.copy()
```

`reset_index()` 또는 `as_index=False`를 사용하면 user가 일반 열이 된다. 확인이나 병합에는 편하지만 모델에 넣기 전 반드시 제거해야 한다.

```python
df_user = df.groupby('user').agg(...).reset_index()
X = df_user.drop(columns='user')
```

`user`는 고객을 구별하기 위한 번호일 뿐 고객의 행동 특성이 아니다. 정규화해서 모델에 넣어도 의미 있는 연속형 변수가 되지 않는다.


## Q03 — 전처리와 K-means 최종 풀이

### 이전 풀이가 29개인데도 틀렸던 이유

수정 전 풀이의 29개 변수에는 `user`가 들어 있고 총 구매금액이 없었다. 올바른 29개 변수에는 총 구매금액이 들어 있고 `user`가 없어야 한다. 현재 풀이에서는 `groupby('user')` 결과를 `reset_index()` 하지 않아 user가 인덱스로 남고, `purchase=('purchase', 'sum')`을 포함하므로 이 문제가 해결되었다.

```text
수정 전: user + gender + age + marital + prod_count + job 21개 + city 3개 = 29
현재 정답: gender + age + marital + prod_count + total_purchase + job 21개 + city 3개 = 29
```

수정 전에는 개수만 같고 변수 하나가 서로 바뀌었기 때문에 shape 검사만으로 발견할 수 없었다. 현재 출력 열에는 `user`가 없고 `purchase`가 있으므로 올바르다. shape와 함께 `X.columns`를 확인하는 습관이 중요하다. 현재 코드의 결과 열 이름 `kind`, `purchase`는 각각 이 노트에서 사용한 설명용 이름 `prod_count`, `total_purchase`와 의미가 같으며, 결과 열 이름 자체는 정답에 영향을 주지 않는다.

### 범주 변환

```python
df_user['gender'] = df_user['gender'].map({'M': 1, 'F': 0})

age_order = ['0-17', '18-25', '26-35', '36-45', '46-50', '51-55', '55+']
age_map = {age: code for code, age in enumerate(age_order)}
df_user['age_group'] = df_user['age_group'].map(age_map)
```

현재 데이터에서는 문자열 정렬 결과도 우연히 올바른 나이 순서가 된다. 하지만 범주 라벨이 바뀌면 문자열 사전순이 실제 나이 순서와 다를 수 있으므로 순서를 명시하는 방식이 안전하다. `replace()`와 `map()` 모두 가능하며, 여기서는 모든 범주의 대응표를 명확히 보여주는 `map()`을 사용했다.

### One Hot Encoding과 변수 개수

```python
X = pd.get_dummies(
    df_user,
    columns=['job', 'city'],
    drop_first=False
)
display(X.shape)
display(X.columns.tolist())
```

직업은 21개 범주, 도시는 3개 범주이므로 더미변수는 24개다. 여기에 `gender`, `age_group`, `marital`, `prod_count`, `total_purchase` 5개를 더하면 총 29개가 된다.

### 군집분석에서는 전체 데이터에 `fit_transform()`

지도학습에서는 평가 데이터 누수를 막기 위해 train에 `fit`, test에 `transform`한다. 이번 문제는 train/test 분할이 없는 비지도 군집분석이며 주어진 5,891명 전체를 군집화한다. 따라서 전체 X에서 scaler를 학습하고 변환하는 `fit_transform()`이 맞다.

```python
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
```

### Silhouette score란?

실루엣 스코어는 각 데이터가 **자신이 속한 군집 안에서는 얼마나 가까이 모여 있고**, **다른 군집과는 얼마나 멀리 떨어져 있는지**를 동시에 평가한다. 정답 라벨이 없는 비지도학습에서 군집 결과의 품질을 내부적으로 평가하는 지표다.

각 고객 `i`에 대해 두 거리를 계산한다.

- `a(i)`: 고객 i와 같은 군집에 속한 다른 고객들까지의 평균 거리 — 군집 내부 응집도
- `b(i)`: 고객 i에서 가장 가까운 다른 군집까지의 평균 거리 — 군집 간 분리도

고객별 실루엣 값은 다음과 같다.

```text
s(i) = (b(i) - a(i)) / max(a(i), b(i))
```

- `a(i)`가 작을수록 같은 군집의 고객들과 가깝게 모여 있다.
- `b(i)`가 클수록 가장 가까운 다른 군집과도 멀리 분리되어 있다.
- 따라서 `a(i)`는 작고 `b(i)`는 클수록 `s(i)`가 1에 가까워진다.

`silhouette_score()`는 5,891명 각각의 `s(i)`를 계산한 뒤 전체 평균을 반환한다.

### 점수 범위와 해석

| 점수 | 의미 |
|---:|---|
| 1에 가까움 | 같은 군집끼리 잘 모이고 다른 군집과 잘 분리됨 |
| 0에 가까움 | 군집 경계가 겹치거나 구분이 뚜렷하지 않음 |
| 음수 | 다른 군집에 더 가까워 잘못 배정되었을 가능성 |

이 문제의 점수 `0.178792... → 0.18`은 군집 간 분리가 강하지 않고 일부 겹침이 있다는 뜻이다. `18% 정확도`라는 의미는 아니다. 실루엣 스코어에는 모든 문제에 공통으로 적용되는 합격 기준이 없으므로 같은 데이터와 전처리 조건에서 다른 k 또는 다른 군집 모델의 점수와 비교해야 한다.

### `silhouette_score(X_scaled, labels)`에 두 인자를 넣는 이유

```python
labels = model.fit_predict(X_scaled)
score = silhouette_score(X_scaled, labels)
```

첫 번째 인자 `X_scaled`는 고객 사이의 거리를 계산할 좌표다.

```text
X_scaled.shape = (5891, 29)
```

각 행은 고객 한 명, 각 열은 정규화된 29개 특성이다. 실루엣 스코어는 기본적으로 이 좌표에서 유클리드 거리를 계산한다. K-means도 유클리드 거리 기반이므로 같은 정규화 데이터를 전달해야 일관된 평가가 된다. 원본 X를 전달하면 구매금액처럼 범위가 큰 변수가 거리를 지배하여 학습한 군집과 다른 기준으로 평가하게 된다.

두 번째 인자 `labels`는 각 고객이 어느 군집에 배정됐는지를 알려준다.

```text
labels.shape = (5891,)
labels 예시 = [4, 5, 4, ..., 2, 1, 6]
```

실루엣 함수는 X만 보고서는 같은 군집과 다른 군집을 구분할 수 없으므로 반드시 군집 라벨도 함께 받아야 한다. `pred_y`라고 이름을 붙여도 계산은 되지만 정답 y를 예측한 것이 아니므로 `labels` 또는 `cluster_labels`가 의미상 더 정확하다.

다음 세 표현은 같은 군집 결과를 평가한다.

```python
# 방법 1
labels = model.fit_predict(X_scaled)
silhouette_score(X_scaled, labels)

# 방법 2
model.fit(X_scaled)
labels = model.predict(X_scaled)
silhouette_score(X_scaled, labels)

# 방법 3
model.fit(X_scaled)
silhouette_score(X_scaled, model.labels_)
```

같은 데이터에 대해 학습과 라벨 생성을 한 번에 수행하는 `fit_predict()`가 가장 간결하다.

### 계산 가능한 조건과 주의점

- 라벨 개수는 최소 2개 이상이어야 한다. 군집이 하나뿐이면 다른 군집까지의 거리 `b(i)`를 계산할 수 없다.
- 각 행이 서로 다른 군집인 경우처럼 군집 수가 표본 수와 같아도 계산할 수 없다.
- 비교할 모델들은 같은 데이터, 같은 전처리, 같은 거리 기준을 사용해야 한다.
- 표본이 많으면 고객 간 거리 계산 비용이 커질 수 있다. 필요하면 `sample_size`로 일부 표본을 평가할 수 있지만, 이 문제는 문제 지시대로 전체 데이터를 사용한다.

### Inertia와의 차이

K-means의 `inertia_`는 각 데이터와 소속 군집 중심 사이의 제곱거리 합으로, 군집 내부가 얼마나 조밀한지만 본다. k를 늘리면 일반적으로 계속 작아지므로 값만 보고 최적 k를 고르기 어렵다. 실루엣 스코어는 군집 내부 응집도와 다른 군집과의 분리도를 함께 보며 **클수록 좋다**.

```python
display(model.inertia_)                    # 작을수록 내부가 조밀
display(silhouette_score(X_scaled, labels)) # 클수록 응집·분리가 좋음
```

### 전체 정답 코드

```python
df_user = df.groupby('user').agg(
    gender=('gender', 'first'),
    age_group=('age_group', 'first'),
    job=('job', 'first'),
    city=('city', 'first'),
    marital=('marital', 'first'),
    prod_count=('prod', 'nunique'),
    total_purchase=('purchase', 'sum')
)

df_user['gender'] = df_user['gender'].map({'M': 1, 'F': 0})

age_order = ['0-17', '18-25', '26-35', '36-45', '46-50', '51-55', '55+']
age_map = {age: code for code, age in enumerate(age_order)}
df_user['age_group'] = df_user['age_group'].map(age_map)

X = pd.get_dummies(
    df_user,
    columns=['job', 'city'],
    drop_first=False
)
display(X.shape)  # (5891, 29)

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

model = KMeans(
    n_clusters=7,
    random_state=123,
    n_init=10
)
labels = model.fit_predict(X_scaled)

answer_q3 = round(silhouette_score(X_scaled, labels), 2)
display(answer_q3)  # 0.18
```

`model.fit(X_scaled)` 후 `model.predict(X_scaled)`를 호출해도 동작하지만, 같은 데이터의 군집 라벨을 바로 구할 때는 `fit_predict()`가 간결하다. `n_init=10`을 명시하면 sklearn 버전에 따른 기본값 차이를 줄일 수 있다.


## 핵심 암기

1. 분석 단위가 거래인지 고객인지 먼저 확인한다. 고객 군집화라면 고객당 한 행으로 집계한다.
2. Named Aggregation은 `결과열=('원본열', '집계함수')` 형식이다.
3. 고정된 고객 속성은 `first`, 상품 종류 수는 `nunique`, 총 구매금액은 `sum`이다.
4. `purchase='first'`는 첫 거래 한 건, `purchase='sum'`은 모든 거래의 총액이다.
5. 집계 함수 문자열의 앞뒤에 공백을 넣으면 `' first'` 같은 존재하지 않는 메서드를 찾아 오류가 발생한다.
6. 고객 식별자 `user`는 모델 입력에서 제외한다.
7. 변수 개수뿐 아니라 실제 열 이름도 확인한다.
8. 고객별 집계 후 직업과 도시는 One Hot Encoding하고 전체 29개 변수를 확인한다.
9. 전체 고객을 군집화하는 비지도학습에서는 전체 X에 `fit_transform()`을 사용할 수 있다.
10. `silhouette_score(X, labels)`의 X는 거리 계산용 좌표이고 labels는 각 행의 군집 번호다.
11. 실루엣 스코어는 -1~1 범위이며 1에 가까울수록 좋고, `0.18`은 정확도가 아니라 군집 응집도와 분리도의 평균 평가값이다.
12. inertia는 작을수록, 실루엣 스코어는 클수록 좋다.
